In [16]:
#creating consumer, pulls data
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [17]:
#deserializer
import json
import sys, os
sys.path.append(os.path.abspath("../src"))
from models import Ride, ride_deserializer

In [18]:
#
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-to-postgres',
    #value_deserializer=lambda x: json.loads(x)
    value_deserializer=ride_deserializer
)

In [19]:
#connecting to postgres
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [23]:
#Read messages and insert into PostgreSQL:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    #pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    #dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    pickup_dt = ride.lpep_pickup_datetime
    dropoff_dt = ride.lpep_dropoff_datetime

    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime, dropoff_datetime, passenger_count, tip_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt, dropoff_dt,
         ride.passenger_count, ride.tip_amount)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...


UndefinedTable: relation "processed_events" does not exist
LINE 1: INSERT INTO processed_events
                    ^
